# Arbol B+

El código desarrollado establece la estructura y el comportamiento de un Árbol B+. A diferencia del Árbol B estándar, esta estructura garantiza que todos los datos residan exclusivamente en los nodos hoja, utilizando los nodos internos únicamente como guías de enrutamiento, lo que optimiza las búsquedas secuenciales y reduce el acceso a disco.

### 1. Estructura Base
La construcción inicia con la clase **`NodoBMas`**, la cual representa la unidad fundamental del árbol. Sus atributos son:
*   **`claves` e `hijos`**: Arreglos dinámicos destinados al almacenamiento ordenado de valores y punteros, respectivamente.
*   **`es_hoja`**: Variable booleana que identifica los nodos del nivel inferior.
*   **`siguiente`**: Puntero horizontal exclusivo de los nodos hoja, esencial para enlazar la base del árbol y conformar una lista simplemente enlazada, facilitando recorridos por rangos.

### 2. Controlador Central y Reglas Estructurales
La clase **`ArbolBMas`** gestiona la jerarquía. El parámetro `orden` determina el número máximo de hijos por nodo. El código aprovecha características nativas de Python (como tuplas, diccionarios y segmentación de listas mediante *slicing*) para prescindir de clases auxiliares innecesarias, logrando una implementación más limpia y eficiente.

### 3. Mecanismo de Búsqueda y Enrutamiento
El algoritmo de búsqueda emplea recorridos iterativos descendentes:
*   Se utiliza `bisect_right` en los nodos internos. En un Árbol B+, si la clave buscada es igual a una clave separadora, la convención dicta que el dato se encuentra en el subárbol derecho. `bisect_right` resuelve esta lógica de enrutamiento en una sola línea.
*   Al alcanzar la hoja, se utiliza `bisect_left` para confirmar la existencia exacta del dato, retornando un diccionario con los metadatos del hallazgo.

### 4. Lógica de Inserción y División Diferenciada
El procedimiento inserta los datos exclusivamente en las hojas y gestiona la saturación mediante dos métodos distintos:
*   **`_dividir_hoja`**: Corta la hoja a la mitad. La clave central se promueve al padre, pero **una copia permanece en la nueva hoja derecha**. Adicionalmente, se actualizan los punteros `siguiente` para mantener la lista enlazada intacta.
*   **`_dividir_interno`**: Corta el nodo guía a la mitad. La clave central asciende al padre y **desaparece del nivel actual**, ya que su única función era enrutar.

### 5. Lógica de Eliminación y Reestructuración
La supresión ocurre únicamente en las hojas. Si un nodo hoja queda por debajo del mínimo permitido, se ejecuta el balanceo (`_balancear`):
*   **Préstamos (`_prestar_de_izquierda`, `_prestar_de_derecha`)**: Si un hermano adyacente cede una clave a una hoja, la clave separadora en el nodo padre **debe actualizarse** copiando el nuevo primer elemento de la hoja derecha. No ocurre un intercambio vertical directo como en el Árbol B tradicional.
*   **Fusión (`_fusionar`):** Si ningún hermano posee claves suficientes, se unifican ambos nodos hoja. El puntero `siguiente` puentea el nodo eliminado, y se remueve definitivamente la clave separadora correspondiente en el nodo padre.

---

### Ejemplo Práctico de Funcionamiento (Árbol B+ de Orden 3)

Se presenta un Árbol B+ inicializado con un orden de 3. Cada nodo alojará un máximo de 2 claves y un mínimo de 1 clave.

#### 1. Inserciones Iniciales
Se introducen los valores `10` y `20`. Ambos se ubican en el nodo raíz, el cual opera como hoja.
*   **Estado del Árbol:**
    ```text
    [Nivel 0 - Hoja] Claves: [10, 20]
    ```

#### 2. División de Hoja por Saturación
Se inserta el valor `30`. La raíz alcanza 3 claves (`[10, 20, 30]`).
Se ejecuta `_dividir_hoja`. El valor `20` se promueve para crear una raíz interna, pero **se conserva una copia** en la hoja derecha.
*   **Estado del Árbol:**
    ```text
    [Nivel 0 - Interno] Claves: [20]
      [Nivel 1 - Hoja] Claves: [10]
      [Nivel 1 - Hoja] Claves: [20, 30]
    ```

#### 3. Expansión Horizontal y División de Nodo Interno
Se inserta el valor `40`. Por enrutamiento, ingresa a la hoja derecha, dejándola saturada (`[20, 30, 40]`).
Se divide nuevamente la hoja; el `30` sube al padre y se conserva abajo.
*   **Estado del Árbol:**
    ```text
    [Nivel 0 - Interno] Claves: [20, 30]
      [Nivel 1 - Hoja] Claves: [10]
      [Nivel 1 - Hoja] Claves: [20]
      [Nivel 1 - Hoja] Claves: [30, 40]
    Hojas enlazadas: [10] -> [20] -> [30, 40]
    ```

#### 4. Eliminación y Fusión de Hojas
Se elimina el valor `10`. La primera hoja queda vacía (`[]`).
El sistema intenta un préstamo, pero el hermano derecho (`[20]`) está en el límite mínimo (1 clave) y no puede ceder. Se activa `_fusionar`:
1.  La hoja vacía se fusiona con la hoja `[20]`.
2.  La clave separadora correspondiente en el padre (`20`) es eliminada permanentemente.
*   **Estado Final del Árbol:**
    ```text
    [Nivel 0 - Interno] Claves: [30]
      [Nivel 1 - Hoja] Claves: [20]
      [Nivel 1 - Hoja] Claves: [30, 40]
    Hojas enlazadas: [20] -> [30, 40]
    ```

In [ ]:
import math
import bisect

class NodoBMas:
    def __init__(self, es_hoja=True):
        self.claves = []
        self.hijos = []
        self.es_hoja = es_hoja
        self.siguiente = None

class ArbolBMas:
    def __init__(self, orden):
        self.orden = orden
        self.raiz = NodoBMas(es_hoja=True)

    def min_claves(self):
        return math.ceil(self.orden / 2) - 1

    def max_claves(self):
        return self.orden - 1

    # BÚSQUEDA
    def buscar(self, clave):
        nodo = self.raiz
        nivel = 0

        while not nodo.es_hoja:
            idx = bisect.bisect_right(nodo.claves, clave)
            nodo = nodo.hijos[idx]
            nivel += 1

        idx = bisect.bisect_left(nodo.claves, clave)

        if idx < len(nodo.claves) and nodo.claves[idx] == clave:
            tipo = "raiz (que tambien es hoja)" if nodo == self.raiz else "hoja"
            return {"clave": clave, "nivel": nivel, "tipo": tipo}

        return None

    # INSERCIÓN
    def insertar(self, clave):
        if self.buscar(clave) is not None:
            print(f"La clave {clave} ya existe, omitiendo.")
            return

        division = self._insertar_recursivo(self.raiz, clave)
        if division:
            clave_sube, nodo_derecho = division
            nueva_raiz = NodoBMas(es_hoja=False)
            nueva_raiz.claves = [clave_sube]
            nueva_raiz.hijos = [self.raiz, nodo_derecho]
            self.raiz = nueva_raiz

        print(f"Clave {clave} insertada.")

    def _insertar_recursivo(self, nodo, clave):
        if nodo.es_hoja:
            idx = bisect.bisect_left(nodo.claves, clave)
            nodo.claves.insert(idx, clave)
            if len(nodo.claves) > self.max_claves():
                return self._dividir_hoja(nodo)
            return None

        idx = bisect.bisect_right(nodo.claves, clave)
        division = self._insertar_recursivo(nodo.hijos[idx], clave)

        if division:
            clave_sube, nodo_derecho = division
            nodo.claves.insert(idx, clave_sube)
            nodo.hijos.insert(idx + 1, nodo_derecho)
            if len(nodo.claves) > self.max_claves():
                return self._dividir_interno(nodo)
        return None

    def _dividir_hoja(self, nodo):
        medio = len(nodo.claves) // 2
        clave_sube = nodo.claves[medio]

        nuevo_nodo = NodoBMas(es_hoja=True)
        nuevo_nodo.claves = nodo.claves[medio:]
        nodo.claves = nodo.claves[:medio]

        nuevo_nodo.siguiente = nodo.siguiente
        nodo.siguiente = nuevo_nodo

        return (clave_sube, nuevo_nodo)

    def _dividir_interno(self, nodo):
        medio = len(nodo.claves) // 2
        clave_sube = nodo.claves[medio]

        nuevo_nodo = NodoBMas(es_hoja=False)
        nuevo_nodo.claves = nodo.claves[medio + 1:]
        nodo.claves = nodo.claves[:medio]

        nuevo_nodo.hijos = nodo.hijos[medio + 1:]
        nodo.hijos = nodo.hijos[:medio + 1]

        return (clave_sube, nuevo_nodo)

    # ELIMINACIÓN
    def eliminar(self, clave):
        if not self.buscar(clave):
            print(f"Clave {clave} no encontrada.")
            return

        self._eliminar_recursivo(self.raiz, clave)

        if not self.raiz.es_hoja and len(self.raiz.claves) == 0:
            self.raiz = self.raiz.hijos[0]
        print(f"Clave {clave} eliminada.")

    def _eliminar_recursivo(self, nodo, clave):
        if nodo.es_hoja:
            idx = bisect.bisect_left(nodo.claves, clave)
            if idx < len(nodo.claves) and nodo.claves[idx] == clave:
                nodo.claves.pop(idx)
                return True
            return False

        idx = bisect.bisect_right(nodo.claves, clave)
        se_elimino = self._eliminar_recursivo(nodo.hijos[idx], clave)

        if se_elimino:
            self._balancear(nodo, idx)
        return se_elimino

    def _balancear(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        if len(hijo.claves) >= self.min_claves():
            return

        if idx_hijo > 0 and len(padre.hijos[idx_hijo - 1].claves) > self.min_claves():
            self._prestar_de_izquierda(padre, idx_hijo)
        elif idx_hijo < len(padre.hijos) - 1 and len(padre.hijos[idx_hijo + 1].claves) > self.min_claves():
            self._prestar_de_derecha(padre, idx_hijo)
        else:
            if idx_hijo > 0:
                self._fusionar(padre, idx_hijo - 1)
            else:
                self._fusionar(padre, idx_hijo)

    def _prestar_de_izquierda(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        hermano_izq = padre.hijos[idx_hijo - 1]

        if hijo.es_hoja:
            hijo.claves.insert(0, hermano_izq.claves.pop())
            padre.claves[idx_hijo - 1] = hijo.claves[0]
        else:
            hijo.claves.insert(0, padre.claves[idx_hijo - 1])
            padre.claves[idx_hijo - 1] = hermano_izq.claves.pop()
            hijo.hijos.insert(0, hermano_izq.hijos.pop())

    def _prestar_de_derecha(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        hermano_der = padre.hijos[idx_hijo + 1]

        if hijo.es_hoja:
            hijo.claves.append(hermano_der.claves.pop(0))
            padre.claves[idx_hijo] = hermano_der.claves[0]
        else:
            hijo.claves.append(padre.claves[idx_hijo])
            padre.claves[idx_hijo] = hermano_der.claves.pop(0)
            hijo.hijos.append(hermano_der.hijos.pop(0))

    def _fusionar(self, padre, idx_izq):
        hijo_izq = padre.hijos[idx_izq]
        hijo_der = padre.hijos[idx_izq + 1]

        if hijo_izq.es_hoja:
            hijo_izq.claves.extend(hijo_der.claves)
            hijo_izq.siguiente = hijo_der.siguiente
            padre.claves.pop(idx_izq)
            padre.hijos.pop(idx_izq + 1)
        else:
            hijo_izq.claves.append(padre.claves.pop(idx_izq))
            hijo_izq.claves.extend(hijo_der.claves)
            hijo_izq.hijos.extend(hijo_der.hijos)
            padre.hijos.pop(idx_izq + 1)

    # VISUALIZACIÓN
    def imprimir(self, nodo=None, nivel=0):
        if nodo is None and nivel == 0:
            nodo = self.raiz
            print("Estructura del Árbol B+:")

        if nodo is None:
            return

        identacion = "  " * nivel
        tipo = "Hoja" if nodo.es_hoja else "Interno"
        print(f"{identacion}[Nivel {nivel} - {tipo}] Claves: {nodo.claves}")

        if not nodo.es_hoja:
            for hijo in nodo.hijos:
                self.imprimir(hijo, nivel + 1)

    def imprimir_hojas(self):
        nodo = self.raiz
        while not nodo.es_hoja:
            nodo = nodo.hijos[0]

        valores = []
        while nodo is not None:
            valores.append(str(nodo.claves))
            nodo = nodo.siguiente

        print("Hojas enlazadas: " + " -> ".join(valores))


# MENÚ
def main():
    while True:
        try:
            orden = int(input("Ingrese el orden del arbol (M >= 3): "))
            if orden >= 3: break
            print("El orden debe ser al menos 3.")
        except ValueError:
            print("Entrada invalida.")

    arbol = ArbolBMas(orden)

    while True:
        print("\n1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir")
        opc = input("Seleccione: ")

        try:
            if opc == '1':
                arbol.insertar(int(input("Clave a insertar: ")))
            elif opc == '2':
                arbol.eliminar(int(input("Clave a eliminar: ")))
            elif opc == '3':
                res = arbol.buscar(int(input("Clave a buscar: ")))
                if res:
                    print(f"Encontrada: {res['clave']} (Nivel {res['nivel']}, Nodo {res['tipo']})")
                else:
                    print("Clave no encontrada.")
            elif opc == '4':
                arbol.imprimir()
                print()
                arbol.imprimir_hojas()
            elif opc == '0':
                break
            else:
                print("Opción inválida.")
        except ValueError:
            print("Por favor ingrese números válidos.")

if __name__ == "__main__":
    main()

Ingrese el grado M del arbol (M >= 3): 3
Cada nodo (menos la raiz) tendra entre 1 y 2 claves, y entre 2 y 3 hijos.

1. Insertar  2. Eliminar  3. Buscar  4. Ver estructura  0. Salir
Opcion: 1
Clave (int): 2
Clave 2 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Ver estructura  0. Salir
Opcion: 1
Clave (int): 4
Clave 4 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Ver estructura  0. Salir
Opcion: 4
Estructura por niveles:
[nivel 0, raiz/hoja] [2, 4]

1. Insertar  2. Eliminar  3. Buscar  4. Ver estructura  0. Salir


KeyboardInterrupt: Interrupted by user